## Task 2: Focal Loss

**Goal:** Use Focal Loss as an alternative to cross-entropy for training a classifier on a strongly imbalanced dataset.

1. Generate a strongly imbalanced binary classification dataset (95% class 0, 5% class 1):

```python
from sklearn.datasets import make_classification

IMBALANCE_RATIO = 0.95

def make_imbalanced_dataset(n_samples=12000, n_features=20, imbalance_ratio=0.95, seed=42):
    """
    Synthetic strongly imbalanced binary problem.
    Some minority class examples overlap with the majority, making classification harder.
    """
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=8,
        n_redundant=6,
        n_repeated=0,
        n_classes=2,
        n_clusters_per_class=2,
        weights=[imbalance_ratio, 1 - imbalance_ratio],
        class_sep=0.9,
        flip_y=0.02,
        random_state=seed,
    )
    return X.astype(np.float32), y.astype(np.float32)
```

2. Implement `BinaryFocalLoss`:

```python
FOCAL_ALPHA = 0.75   # weight for the positive class
FOCAL_GAMMA = 2.0    # focusing on hard examples

class BinaryFocalLoss(nn.Module):
    """
    Focal Loss for binary classification.
    FL = -alpha_t * (1 - p_t)^gamma * BCE
    where:
      p_t     = p if y=1, 1-p if y=0
      alpha_t = weight for the class, e.g. higher for the rare class
    """
    def __init__(self, alpha=0.75, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs   = torch.sigmoid(logits)
        pt      = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss    = alpha_t * (1 - pt).pow(self.gamma) * bce
        if self.reduction == "mean": return loss.mean()
        elif self.reduction == "sum": return loss.sum()
        return loss
```

3. Implement datasets, dataloaders, an MLP classifier, a training loop, and a prediction function. Use the best checkpoint from validation for prediction.

4. Implement evaluation metrics:

```python
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_true, y_prob),
        "pr_auc":    average_precision_score(y_true, y_prob),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }
```

5. Train with `nn.BCEWithLogitsLoss` and with `BinaryFocalLoss`. Compare results.

**Assignment:** Implement and train a classifier on a strongly imbalanced dataset — compare classification quality for cross-entropy and Focal Loss.


In [1]:
from utils import make_imbalanced_dataset, BinaryFocalLoss
import torch

IMBALANCE_RATIO = 0.95
FOCAL_ALPHA = 0.75   # weight for the positive class
FOCAL_GAMMA = 2.0    # focusing on hard examples

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

X, y = make_imbalanced_dataset(imbalance_ratio=IMBALANCE_RATIO)
model = BinaryFocalLoss().to(device)